# 🧠 Logic Fine-Tuning + Z3 Hybrid Pipeline: Qwen3-8B × Unsloth
**Pipeline:** Data Unpacking → LoRA Training → Z3-Verified Inference → Neural CoT Fallback

> Dataset: `Logic_Based_Educational_Queries-2.json` | 411 indices | FOL + NL premises  
> Target: **≥70% accuracy** trên toàn bộ test set

---
## 🔧 Kiến trúc Hybrid 5 Tầng (v2 — Updated)

```
INPUT
  │
  ▼ STAGE 1: Formal Ontology Compiler
  │   • Extract predicate names + ARITY từ premises-FOL (100% algorithmic)
  │   • Resolve arity conflicts, sinh sẵn Z3 declaration block
  │
  ▼ STAGE 2: Constrained FOL Translation (per MCQ option)
  │   • Tách riêng question stem + 4 options A/B/C/D
  │   • Qwen chỉ dịch MỘT option (target statement) → Z3 QUERY
  │   • Declaration block từ Stage 1 được inject sẵn (Qwen không tự khai báo)
  │
  ▼ STAGE 3: Z3 Multi-Target Entailment (loop qua từng option)
  │
  ├──[ENTAILED]──▶ ĐÁP ÁN CHẮC CHẮN (dừng sớm)
  │
  └──[FAIL/UNKNOWN]──▶ STAGE 4: Semantic Self-Correction Loop
          │  • Syntax error → Z3 error feedback
          │  • Unknown result → Semantic feedback (quantifier/direction hint)
          │
          [FIX OK]──▶ ĐÁP ÁN TỪ Z3
                 │
          [STILL FAIL]──▶ STAGE 5: CoT Neural Fallback + Majority Vote
                              • stop_strings guard chống token explosion
```

---
## 📋 Các thay đổi v2 so với v1
| # | Thay đổi | Mô tả |
|---|----------|-------|
| 1 | **MCQ Per-Option Loop** | Tách stem/options; duyệt từng option qua Z3; dừng sớm khi entailed |
| 2+3 | **Dynamic Arity Extraction** | Stage 1 extract arity thực từ FOL; sinh Z3 declaration block; inject vào Stage 2 |
| 4 | **Semantic Self-Correction** | Stage 4 phân biệt syntax error vs semantic unknown; feedback khác nhau |
| 5 | **CoT Stop Strings** | `stop_strings` guard trong `model.generate()` chống infinite thinking loop |


In [ ]:
%%capture
# Unsloth với CUDA 12.1 (T4 / A100 Kaggle)
!pip install unsloth
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes
!pip install datasets transformers sentencepiece protobuf
!pip install z3-solver   # ← Z3 theorem prover


In [ ]:
import json
import os
import re
import random
import copy
import ast
import textwrap
from collections import Counter, defaultdict
from typing import List, Dict, Any, Optional, Tuple
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Z3
from z3 import (
    Solver, Bool, BoolRef, sat, unsat, unknown,
    ForAll, Exists, And, Or, Not, Implies, Function,
    IntSort, BoolSort, DeclareSort, Const, Consts,
    set_param
)
set_param('timeout', 5000)   # 5s timeout cho mỗi Z3 query

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
try:
    import z3; print(f"✅ Z3 version: {z3.get_version_string()}")
except: print("⚠️  Z3 not installed yet, sẽ có sau install")


In [ ]:
# ============================================================
#  GLOBAL CONFIG
# ============================================================
DATA_PATH  = "/kaggle/input/datasets/barakuga/exact-dataset/Logic_Based_Educational_Queries-2.json"
OUTPUT_DIR = "/kaggle/working/lora_output_qwen3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME  = "unsloth/Qwen3-8B-bnb-4bit"
MAX_SEQ_LEN = 8192

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

LEARNING_RATE    = 1e-4
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0

TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
UNKNOWN_OVERSAMPLE_FACTOR = 2

# Z3 Pipeline Config
Z3_BON          = 3      # Best-of-N: số lần dịch FOL
Z3_MAX_RETRIES  = 1      # Số lần self-correction tối đa
THINKING_BUDGET = 512    # Token tối đa cho thinking block của Qwen3

print("✅ Config loaded.")
print(f"   Model: {MODEL_NAME} | LoRA r={LORA_R} | LR={LEARNING_RATE}")
print(f"   Z3 Best-of-N={Z3_BON} | Retries={Z3_MAX_RETRIES}")


In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data: List[Dict] = json.load(f)

print(f"📦 Tổng số indices: {len(raw_data)}")
sample = raw_data[0]
print(f"\n--- Cấu trúc 1 index mẫu ---")
print(f"  idx            : {sample['idx']}")
print(f"  premises-FOL   : {len(sample['premises-FOL'])} premises")
print(f"  premises-NL    : {len(sample['premises-NL'])} premises")
print(f"  questions      : {len(sample['questions'])} questions")
print(f"  answers        : {sample['answers']}")

all_answers = [a for item in raw_data for a in item["answers"]]
answer_dist = Counter(all_answers)
print(f"\n📊 Phân phối nhãn ({len(all_answers)} samples):")
for label, count in sorted(answer_dist.items()):
    pct = count / len(all_answers) * 100
    print(f"  {label:10s}: {count:4d} ({pct:5.1f}%) {'█'*int(pct/2)}")


In [ ]:
# ============================================================
#  PROMPT TEMPLATES — Training (CoT) + Z3 (FOL)
# ============================================================

SYSTEM_PROMPT = """You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).
Given a set of premises (in natural language and FOL notation), you must:
1. Analyze the logical structure of each premise carefully.
2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation, existential instantiation, hypothetical syllogism.
3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.
4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.
5. For Yes/No questions: verify the statement logically.
6. If the premises are INSUFFICIENT to determine a unique conclusion, your Final Answer MUST be exactly: Unknown
7. Never hallucinate a conclusion that is not entailed by the premises.
Format your response EXACTLY as:
### Step-by-Step Reasoning
<your reasoning here>
### Final Answer
<A or B or C or D or Yes or No or Unknown>"""


# ── MCQ Parsing ─────────────────────────────────────────────
_MCQ_OPTION_RE = re.compile(
    r'(?:^|\n)\s*\(?([A-D])[.)\s]\s*(.+?)(?=\n\s*\(?[A-D][.)\s]|$)',
    re.DOTALL
)

def parse_mcq(question: str):
    """
    Tách question thành (stem, {A: text, B: text, C: text, D: text}).
    Nếu không tìm được options → trả về (question, {}) để fallback CoT.
    """
    options = {}
    for m in _MCQ_OPTION_RE.finditer(question):
        label = m.group(1).upper()
        text  = m.group(2).strip()
        options[label] = text

    if options:
        # stem là phần trước option đầu tiên
        first_match = _MCQ_OPTION_RE.search(question)
        stem = question[:first_match.start()].strip() if first_match else question
    else:
        stem = question

    return stem, options


# ── Z3 FOL Translation Prompt (v2: arity-aware, declaration injected) ──
def make_fol_translation_prompt(
    nl_premises: List[str],
    fol_premises: List[str],
    ontology_info: Dict,          # {name: {arity: int, desc: str}}
    z3_declaration_block: str,    # pre-generated by Stage 1
    target_statement: str,
    error_feedback: str = "",
    feedback_type: str = "syntax"  # "syntax" | "semantic"
) -> str:
    """
    Stage 2: Prompt ép Qwen dịch target_statement sang Z3 QUERY.
    Declaration block đã được Stage 1 sinh sẵn — Qwen KHÔNG được khai báo lại.
    """
    # Hiển thị tường minh tên + arity cho Qwen
    pred_lines = []
    for name, info in ontology_info.items():
        arity = info["arity"]
        args  = ", ".join([f"arg{i+1}" for i in range(arity)])
        pred_lines.append(f"  - {name}({args})  # {info['desc']}")
    predicate_list = "\n".join(pred_lines)

    premise_block = "\n".join(
        f"P{i+1}. [NL] {nl}\n   [FOL] {fol}"
        for i, (nl, fol) in enumerate(zip(nl_premises, fol_premises))
    )

    error_section = ""
    if error_feedback:
        if feedback_type == "syntax":
            error_section = f"""
⚠️  PREVIOUS ATTEMPT FAILED — Z3 Syntax Error:
{error_feedback}
Common fixes:
- Use Implies(a, b) not (a >> b)
- Use And(a, b) not (a & b)
- ForAll([x], body) needs list syntax
- Do NOT re-declare predicates — they are already declared above
"""
        else:  # semantic
            error_section = f"""
⚠️  PREVIOUS ATTEMPT — Semantic Issue (code ran but Z3 returned UNKNOWN):
The solver could not find an entailment proof. Likely causes:
- Implication translated in the wrong direction (use Implies(cause, effect), not reversed)
- Wrong quantifier: check whether ForAll or Exists is appropriate
- Missing transitive chain — make sure all intermediate steps are encoded
Please rewrite the QUERY formula more carefully.
"""

    return f"""You are a formal logic expert. Translate the TARGET STATEMENT to a Z3 QUERY expression.

CRITICAL RULES:
1. The declaration block below is ALREADY COMPLETE. Do NOT re-declare Entity, predicates, or variables.
2. Output ONLY the body code (assertions + PREMISES list + QUERY assignment). No markdown fences.
3. ONLY use predicates from the approved list — exact name and arity.
4. End with: PREMISES = [p1, p2, ...] and QUERY = <formula>

PRE-DECLARED (do not repeat these lines):
{z3_declaration_block}

APPROVED PREDICATES (name and arity — use exactly):
{predicate_list}

PREMISES:
{premise_block}

TARGET STATEMENT TO ENCODE AS QUERY:
{target_statement}
{error_section}
WRITE THE BODY CODE NOW (no declarations, no markdown):"""


def make_cot_prompt(fol_list, nl_list, question):
    """Stage 5 fallback: CoT prompt chuẩn để Qwen suy luận ngôn ngữ tự nhiên."""
    lines = ["### Premises"]
    for i, (fol, nl) in enumerate(zip(fol_list, nl_list), 1):
        lines.append(f"P{i}. [NL]  {nl}")
        lines.append(f"   [FOL] {fol}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"


def build_assistant_message(explanation: str, answer: str) -> str:
    return (
        f"### Step-by-Step Reasoning\n{explanation}\n\n"
        f"### Final Answer\n{answer}"
    )

print("✅ Prompt templates defined (v2: arity-aware, MCQ parser, semantic feedback).")

# Smoke-test MCQ parser
_test_q = """Which of the following is true?
A) All birds can fly.
B) Some mammals are reptiles.
C) All humans are mortal.
D) No fish can swim."""
_stem, _opts = parse_mcq(_test_q)
assert set(_opts.keys()) == {"A","B","C","D"}, f"MCQ parse failed: {_opts}"
print(f"   MCQ parser test ✅ — stem='{_stem[:40]}...', options={list(_opts.keys())}")


In [ ]:
# ============================================================
#  STAGE 1: Formal Ontology Compiler — Dynamic Arity Extraction
# ============================================================

_FOL_BUILTINS = frozenset({
    "ForAll", "Exists", "And", "Or", "Not", "Implies",
    "Iff", "Xor", "If", "Then", "Else"
})

# Fallback NL keywords khi FOL premises rỗng
_NL_PREDICATE_KEYWORDS = {
    "student": "Student", "teacher": "Teacher", "professor": "Professor",
    "person": "Person", "human": "Human", "animal": "Animal",
    "bird": "Bird", "mammal": "Mammal", "dog": "Dog", "cat": "Cat",
    "pass": "PassExam", "fail": "FailExam", "study": "Studies",
    "learn": "Learns", "know": "Knows", "smart": "Smart",
    "intelligent": "Intelligent", "happy": "Happy", "sad": "Sad",
    "work": "Works", "play": "Plays", "run": "Runs", "fly": "Flies",
    "swim": "Swims", "eat": "Eats", "drink": "Drinks",
    "love": "Loves", "hate": "Hates", "like": "Likes",
    "old": "Old", "young": "Young", "tall": "Tall", "short": "Short",
    "rich": "Rich", "poor": "Poor", "sick": "Sick",
    "healthy": "Healthy", "mortal": "Mortal", "dead": "Dead",
    "alive": "Alive", "win": "Wins", "lose": "Loses",
}


def _extract_arity_from_fol_string(fol: str) -> Dict[str, int]:
    """
    Quét một chuỗi FOL, trả về {predicate_name: arity} cho tất cả
    lời gọi hàm tìm thấy (loại bỏ FOL builtins).
    Arity = số argument phân tách bởi dấu phẩy bên trong ngoặc đơn.
    """
    result = {}
    # Match: CapWord( ... )  — lấy nội dung trong ngoặc đầu tiên
    for m in re.finditer(r'\b([A-Z][a-zA-Z0-9]*)\s*\(([^)]*)\)', fol):
        name = m.group(1)
        if name in _FOL_BUILTINS:
            continue
        args_str = m.group(2).strip()
        if not args_str:
            arity = 0
        else:
            # Đếm argument bằng cách split theo dấu phẩy
            # (đủ chính xác với FOL đơn giản; không lồng ngoặc phức tạp)
            arity = len([a for a in args_str.split(",") if a.strip()])
        # Conflict resolution: lấy arity max nếu cùng tên xuất hiện nhiều lần
        if name in result:
            if result[name] != arity:
                print(f"   ⚠️  Arity conflict for '{name}': {result[name]} vs {arity} → taking max")
                result[name] = max(result[name], arity)
        else:
            result[name] = arity
    return result


def build_ontology_v2(
    nl_premises: List[str],
    fol_premises: List[str]
) -> Tuple[Dict[str, Dict], str]:
    """
    Stage 1 (v2): Formal Ontology Compiler.

    Returns:
        ontology_info : {name: {arity: int, desc: str}}
        z3_decl_block : str  — Python code khai báo sẵn Entity sort + tất cả predicates
                               để inject vào prompt Stage 2 và sandbox.
    """
    arity_map: Dict[str, int] = {}

    # --- 1. Extract từ FOL premises (nguồn chính xác nhất) ---
    for fol in fol_premises:
        for name, arity in _extract_arity_from_fol_string(fol).items():
            if name in arity_map and arity_map[name] != arity:
                print(f"   ⚠️  Cross-premise conflict '{name}': {arity_map[name]} vs {arity} → max")
                arity_map[name] = max(arity_map[name], arity)
            else:
                arity_map[name] = arity

    # --- 2. Fallback sang NL keyword matching nếu FOL rỗng ---
    if not arity_map:
        all_nl = " ".join(nl_premises).lower()
        for keyword, pred_name in _NL_PREDICATE_KEYWORDS.items():
            if keyword in all_nl and pred_name not in arity_map:
                arity_map[pred_name] = 1  # NL fallback luôn assume unary

    # --- 3. Hardcoded safety net ---
    if not arity_map:
        arity_map["P"] = 1
        arity_map["Q"] = 1

    # --- 4. Build ontology_info dict ---
    ontology_info = {}
    for name, arity in arity_map.items():
        desc = f"{arity}-ary predicate from FOL"
        ontology_info[name] = {"arity": arity, "desc": desc}

    # --- 5. Generate Z3 declaration block ---
    decl_lines = [
        "Entity = DeclareSort('Entity')",
        "x = Const('x', Entity)",
        "y = Const('y', Entity)",
        "z = Const('z', Entity)",
    ]
    for name, arity in sorted(arity_map.items()):
        sorts = ", ".join(["Entity"] * arity + ["BoolSort()"])
        decl_lines.append(f"{name} = Function('{name}', {sorts})")

    z3_decl_block = "\n".join(decl_lines)

    return ontology_info, z3_decl_block


# ── Test ────────────────────────────────────────────────────
sample = raw_data[0]
test_info, test_decl = build_ontology_v2(sample["premises-NL"], sample["premises-FOL"])
print(f"✅ build_ontology_v2() hoạt động.")
print(f"   Số vị từ extracted: {len(test_info)}")
for name, info in list(test_info.items())[:6]:
    print(f"   {name}(arity={info['arity']}): {info['desc']}")
print(f"\n   Z3 declaration block preview:")
for line in test_decl.split("\n")[:6]:
    print(f"   {line}")


In [ ]:
# ============================================================
#  STAGE 2-3: FOL Translation + Z3 Verification Engine (v2)
# ============================================================

class FOLTranslationError(Exception):
    pass

class Z3VerificationError(Exception):
    pass


def run_z3_sandbox(
    body_code: str,
    z3_decl_block: str,
    timeout_ms: int = 5000
) -> Tuple[str, str]:
    """
    Chạy Z3 sandbox với declaration block được inject sẵn từ Stage 1.
    body_code: chỉ chứa assertions + PREMISES + QUERY (Qwen sinh ra).
    z3_decl_block: khai báo Entity sort + predicates từ build_ontology_v2().

    Returns: (result_str, error_str)
    result_str ∈ {"entailed", "contradicted", "unknown", "error"}
    """
    full_code = z3_decl_block + "\n" + body_code

    namespace = {
        'Solver': Solver, 'Bool': Bool, 'sat': sat, 'unsat': unsat, 'unknown': unknown,
        'ForAll': ForAll, 'Exists': Exists, 'And': And, 'Or': Or, 'Not': Not,
        'Implies': Implies, 'Function': Function, 'IntSort': IntSort,
        'BoolSort': BoolSort, 'DeclareSort': DeclareSort, 'Const': Const, 'Consts': Consts,
        'set_param': set_param,
    }
    namespace['set_param']('timeout', timeout_ms)

    try:
        exec(compile(full_code, '<z3_sandbox>', 'exec'), namespace)
    except SyntaxError as e:
        return "error", f"SyntaxError: {e}"
    except Exception as e:
        return "error", f"RuntimeError: {type(e).__name__}: {e}"

    premises = namespace.get('PREMISES', None)
    query    = namespace.get('QUERY', None)

    if premises is None or query is None:
        return "error", "Code must define PREMISES = [...] and QUERY = <formula>"

    # Entailment via refutation: premises ⊨ query  ⟺  premises ∧ ¬query is UNSAT
    s = Solver()
    try:
        for p in premises:
            s.add(p)
        s.add(Not(query))
        result = s.check()
        if result == unsat:
            return "entailed", ""
        elif result == sat:
            s2 = Solver()
            for p in premises:
                s2.add(p)
            s2.add(query)
            result2 = s2.check()
            if result2 == unsat:
                return "contradicted", ""
            else:
                return "unknown", ""
        else:
            return "unknown", "Z3 timeout or undecidable"
    except Exception as e:
        return "error", f"Z3 check error: {e}"


def extract_z3_code(raw_output: str) -> str:
    """Trích Z3 body code từ output của Qwen (strip markdown + thinking tags)."""
    code = raw_output.strip()
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0]
    elif "```" in code:
        code = code.split("```")[1].split("```")[0]
    if "<think>" in code:
        code = re.sub(r'<think>.*?</think>', '', code, flags=re.DOTALL).strip()
    return code.strip()


print("✅ Z3 sandbox engine ready (v2: declaration block injected externally).")
print("   Testing Z3 sandbox v2...")

# Test với declaration block tách riêng
_test_decl = """Entity = DeclareSort('Entity')
x = Const('x', Entity)
y = Const('y', Entity)
Student = Function('Student', Entity, BoolSort())
Passes = Function('Passes', Entity, BoolSort())"""

_test_body = """
alice = Const('alice', Entity)
p1 = ForAll([x], Implies(Student(x), Passes(x)))
p2 = Student(alice)
PREMISES = [p1, p2]
QUERY = Passes(alice)
"""
_res, _err = run_z3_sandbox(_test_body, _test_decl)
print(f"   Test result: {_res} | Error: '{_err}'")
assert _res == "entailed", f"Z3 sandbox v2 test FAILED: {_res}, {_err}"
print("✅ Z3 sandbox v2 test passed!")


In [ ]:
# ============================================================
#  FULL 5-STAGE INFERENCE PIPELINE (v2)
#  Fix 1: MCQ per-option loop
#  Fix 2+3: Arity-aware ontology, declaration injection
#  Fix 4: Semantic self-correction (syntax vs unknown)
#  Fix 5: stop_strings guard for CoT
# ============================================================

# CoT stop strings — ngắt ngay khi mô hình chốt đáp án cuối
_COT_STOP_STRINGS = [
    "### Final Answer\nA", "### Final Answer\nB",
    "### Final Answer\nC", "### Final Answer\nD",
    "### Final Answer\nYes", "### Final Answer\nNo",
    "### Final Answer\nUnknown",
    "### Final Answer:\nA", "### Final Answer:\nB",
    "### Final Answer:\nC", "### Final Answer:\nD",
    "### Final Answer:\nYes", "### Final Answer:\nNo",
    "### Final Answer:\nUnknown",
]


def call_qwen(
    model, tokenizer, messages: List[Dict],
    max_new_tokens: int = 512,
    temperature: float = 0.0,
    enable_thinking: bool = False,
    use_stop_strings: bool = False
) -> str:
    """
    Core Qwen3 call.
    enable_thinking=False: inject /no_think → tránh token explosion.
    use_stop_strings=True: kích hoạt stop_strings guard cho CoT (Fix 5).
    """
    if not enable_thinking:
        msgs = []
        for m in messages:
            if m['role'] == 'system':
                msgs.append({'role': 'system', 'content': m['content'] + "\n/no_think"})
            else:
                msgs.append(m)
    else:
        msgs = messages

    input_ids = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    generate_kwargs = dict(
        max_new_tokens     = max_new_tokens,
        temperature        = temperature if temperature > 0 else None,
        do_sample          = temperature > 0,
        repetition_penalty = 1.0,
        pad_token_id       = tokenizer.eos_token_id,
        eos_token_id       = tokenizer.eos_token_id,
    )

    # Fix 5: stop_strings — chỉ áp dụng cho CoT fallback
    if use_stop_strings:
        generate_kwargs["stop_strings"]  = _COT_STOP_STRINGS
        generate_kwargs["tokenizer"]     = tokenizer

    with torch.no_grad():
        output_ids = model.generate(input_ids, **generate_kwargs)

    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# ── Stage 2: translate ONE target statement → Z3 body code ──
def stage2_translate_fol(
    model, tokenizer,
    nl_premises, fol_premises,
    ontology_info: Dict, z3_decl_block: str,
    target_statement: str,
    error_feedback: str = "",
    feedback_type: str = "syntax",
    temperature: float = 0.3
) -> str:
    """Stage 2 (v2): Qwen dịch target_statement → Z3 body code.
    Declaration block đã được Stage 1 sinh — Qwen không tự khai báo.
    """
    prompt = make_fol_translation_prompt(
        nl_premises, fol_premises,
        ontology_info, z3_decl_block,
        target_statement, error_feedback, feedback_type
    )
    messages = [
        {"role": "system", "content": "You are a formal logic expert. Output only the Z3 Python body code. No markdown, no declarations, no explanations."},
        {"role": "user",   "content": prompt},
    ]
    raw = call_qwen(
        model, tokenizer, messages,
        max_new_tokens=400, temperature=temperature,
        enable_thinking=False
    )
    return extract_z3_code(raw)


def stage3_z3_verify(body_code: str, z3_decl_block: str) -> Tuple[str, str]:
    """Stage 3 (v2): Chạy Z3 với declaration block injected."""
    return run_z3_sandbox(body_code, z3_decl_block)


def stage4_self_correct(
    model, tokenizer,
    nl_premises, fol_premises,
    ontology_info: Dict, z3_decl_block: str,
    target_statement: str,
    error_msg: str,
    feedback_type: str  # "syntax" hoặc "semantic"
) -> Tuple[str, str, str]:
    """
    Stage 4 (v2): Self-Correction với feedback phân biệt syntax/semantic.
    Returns (fixed_code, z3_result, new_error)
    """
    fixed_code = stage2_translate_fol(
        model, tokenizer,
        nl_premises, fol_premises,
        ontology_info, z3_decl_block,
        target_statement,
        error_feedback=error_msg,
        feedback_type=feedback_type,
        temperature=0.2
    )
    result, err = stage3_z3_verify(fixed_code, z3_decl_block)
    return fixed_code, result, err


def stage5_cot_fallback(
    model, tokenizer,
    fol_list, nl_list, question: str,
    n_votes: int = 3
) -> Tuple[str, List[str]]:
    """
    Stage 5 (v2): Neural CoT Fallback + Majority Vote.
    Fix 5: dùng stop_strings để ngắt ngay khi model chốt đáp án.
    """
    cot_prompt = make_cot_prompt(fol_list, nl_list, question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": cot_prompt},
    ]

    votes = []
    for i in range(n_votes):
        temp = 0.3 if i > 0 else 0.0
        raw = call_qwen(
            model, tokenizer, messages,
            max_new_tokens=600,
            temperature=temp,
            enable_thinking=True,
            use_stop_strings=True   # Fix 5: stop ngay sau Final Answer
        )
        ans = extract_final_answer_cot(raw)
        votes.append(ans)

    vote_counter = Counter(v for v in votes if v != "UNPARSEABLE")
    winner = vote_counter.most_common(1)[0][0] if vote_counter else "Unknown"
    return winner, votes


def z3_result_to_yesno(z3_result: str) -> Optional[str]:
    """Convert Z3 result → Yes/No/Unknown cho dạng câu Yes-No."""
    if z3_result == "entailed":    return "Yes"
    if z3_result == "contradicted": return "No"
    if z3_result == "unknown":      return "Unknown"
    return None


def run_hybrid_pipeline(
    model, tokenizer,
    entry: Dict, question_idx: int
) -> Tuple[str, str]:
    """
    Full 5-stage pipeline cho 1 question (v2).
    Fix 1: MCQ per-option loop — duyệt A/B/C/D độc lập, dừng khi entailed.
    Fix 2+3: Arity-aware Stage 1 + declaration injection.
    Fix 4: Semantic self-correction phân biệt syntax vs unknown.
    Fix 5: stop_strings trong Stage 5.

    Returns: (final_answer, route_taken)
    """
    nl_premises  = entry["premises-NL"]
    fol_premises = entry["premises-FOL"]
    question     = entry["questions"][question_idx]

    # ── STAGE 1: Formal Ontology Compiler (Fix 2+3) ─────────────
    ontology_info, z3_decl_block = build_ontology_v2(nl_premises, fol_premises)

    # ── Phân loại câu hỏi ───────────────────────────────────────
    stem, mcq_options = parse_mcq(question)
    is_mcq   = bool(mcq_options)
    is_yesno = (not is_mcq) and any(
        question.lower().strip().startswith(p) for p in
        ["is ", "are ", "does ", "do ", "can ", "will ", "has ", "have ",
         "was ", "were ", "did ", "should ", "would ", "could "]
    )

    # ══════════════════════════════════════════════════════════════
    # FIX 1: MCQ PER-OPTION LOOP
    # Duyệt từng option A→D; dừng ngay khi Z3 chứng minh entailed
    # ══════════════════════════════════════════════════════════════
    if is_mcq:
        for option_label in ["A", "B", "C", "D"]:
            if option_label not in mcq_options:
                continue
            option_text = mcq_options[option_label]
            # Target statement = stem + option content
            target = f"{stem} → Specifically: {option_text}"

            # Best-of-N Z3 attempts cho option này
            last_error   = ""
            last_result  = ""
            z3_succeeded = False

            for attempt in range(Z3_BON):
                temp = 0.0 if attempt == 0 else 0.4
                body_code = stage2_translate_fol(
                    model, tokenizer,
                    nl_premises, fol_premises,
                    ontology_info, z3_decl_block,
                    target, temperature=temp
                )
                result, err = stage3_z3_verify(body_code, z3_decl_block)
                last_result = result
                last_error  = err if err else last_error

                if result == "entailed":
                    return option_label, f"z3_mcq_option_{option_label}"

                if result in ("contradicted", "unknown"):
                    # Fix 4: kích hoạt semantic correction nếu unknown
                    if result == "unknown":
                        _, corrected, new_err = stage4_self_correct(
                            model, tokenizer,
                            nl_premises, fol_premises,
                            ontology_info, z3_decl_block,
                            target, error_msg="Result was UNKNOWN — check quantifiers/direction.",
                            feedback_type="semantic"
                        )
                        if corrected == "entailed":
                            return option_label, f"z3_mcq_semantic_corrected_{option_label}"
                    break

                if err and result == "error":
                    # Fix 4: syntax correction
                    _, corrected, new_err = stage4_self_correct(
                        model, tokenizer,
                        nl_premises, fol_premises,
                        ontology_info, z3_decl_block,
                        target, error_msg=err,
                        feedback_type="syntax"
                    )
                    if corrected == "entailed":
                        return option_label, f"z3_mcq_syntax_corrected_{option_label}"
                    last_error = new_err if new_err else last_error
                    break

        # Không option nào entailed → CoT fallback
        ans, votes = stage5_cot_fallback(
            model, tokenizer, fol_premises, nl_premises, question
        )
        return ans, f"cot_fallback_mcq({','.join(votes)})"

    # ══════════════════════════════════════════════════════════════
    # YES-NO & UNKNOWN QUESTIONS: Best-of-N như cũ
    # ══════════════════════════════════════════════════════════════
    target = question
    z3_results = []
    last_error = ""

    for attempt in range(Z3_BON):
        temp = 0.0 if attempt == 0 else 0.4
        body_code = stage2_translate_fol(
            model, tokenizer,
            nl_premises, fol_premises,
            ontology_info, z3_decl_block,
            target, temperature=temp
        )
        result, err = stage3_z3_verify(body_code, z3_decl_block)
        z3_results.append(result)
        last_error = err if err else last_error
        if result in ("entailed", "contradicted"):
            break

    meaningful     = [r for r in z3_results if r in ("entailed", "contradicted", "unknown")]
    result_counts  = Counter(meaningful)

    if result_counts:
        top_result, top_count = result_counts.most_common(1)[0]
        if top_count >= max(1, len(meaningful) // 2 + 1):
            # Fix 4: semantic correction trước khi accept unknown
            if top_result == "unknown" and is_yesno:
                _, corrected, _ = stage4_self_correct(
                    model, tokenizer,
                    nl_premises, fol_premises,
                    ontology_info, z3_decl_block,
                    target,
                    error_msg="Result was UNKNOWN — check quantifiers/direction.",
                    feedback_type="semantic"
                )
                if corrected in ("entailed", "contradicted"):
                    ans = z3_result_to_yesno(corrected)
                    return ans, "z3_semantic_corrected"
            ans = z3_result_to_yesno(top_result)
            if ans is not None:
                return ans, "z3_consensus"

    # Fix 4: syntax self-correction nếu có error
    if last_error and "error" in z3_results:
        for _ in range(Z3_MAX_RETRIES):
            _, corrected_result, new_err = stage4_self_correct(
                model, tokenizer,
                nl_premises, fol_premises,
                ontology_info, z3_decl_block,
                target, error_msg=last_error,
                feedback_type="syntax"
            )
            ans = z3_result_to_yesno(corrected_result)
            if ans is not None:
                return ans, "z3_syntax_corrected"
            last_error = new_err if new_err else last_error

    # Stage 5: CoT Neural Fallback
    ans, votes = stage5_cot_fallback(
        model, tokenizer, fol_premises, nl_premises, question
    )
    return ans, f"cot_fallback({','.join(votes)})"


print("✅ Full 5-stage pipeline v2 defined.")
print(f"   Z3 Best-of-N = {Z3_BON}")
print(f"   Self-correction retries = {Z3_MAX_RETRIES}")
print(f"   CoT fallback votes = 3 | stop_strings = {len(_COT_STOP_STRINGS)} patterns")


In [ ]:
# ============================================================
#  ANSWER EXTRACTION — Dùng cho cả Z3 pipeline và CoT
# ============================================================

def extract_final_answer_cot(model_output: str) -> str:
    """
    Multi-pattern extraction từ CoT output.
    Priority: ### Final Answer → inline patterns → fallback
    """
    # Strip thinking tags
    text = re.sub(r'<think>.*?</think>', '', model_output, flags=re.DOTALL).strip()
    
    # Pattern 1: ### Final Answer block
    match = re.search(r'###\s*Final\s*Answer[:\s]*\n?\s*(.+)', text, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        for label in ["unknown", "yes", "no"]:
            if ans.lower().startswith(label):
                return label.capitalize()
        m = re.match(r'^([A-D])[.)\s:]', ans, re.IGNORECASE)
        if m: return m.group(1).upper()
        if re.match(r'^[A-D]$', ans, re.IGNORECASE): return ans.upper()
    
    # Pattern 2: "answer is X" / "answer: X"
    match2 = re.search(r'(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)', text, re.IGNORECASE)
    if match2:
        g = match2.group(1).lower()
        if g == "unknown": return "Unknown"
        if g == "yes":     return "Yes"
        if g == "no":      return "No"
        return g.upper()
    
    # Pattern 3: standalone letter near end
    last_section = text[-500:]
    letters = re.findall(r'\b([A-D])\b', last_section)
    if letters: return letters[-1].upper()
    
    # Pattern 4: keyword anywhere
    for kw, val in [("unknown","Unknown"), ("yes","Yes"), ("no","No")]:
        if re.search(f'\\b{kw}\\b', text, re.IGNORECASE):
            return val
    
    return "UNPARSEABLE"


# Test
test_cases = [
    ("### Final Answer\nA", "A"),
    ("### Final Answer\nUnknown", "Unknown"),
    ("### Final Answer\nYes", "Yes"),
    ("the answer is C", "C"),
    ("### Final Answer\nB. Because...", "B"),
    ("<think>long thinking</think>\n### Final Answer\nNo", "No"),
]
print("Testing extract_final_answer_cot:")
for text, expected in test_cases:
    got = extract_final_answer_cot(text)
    status = "✅" if got == expected else "❌"
    print(f"  {status} '{text[:50]}'  → got={got}, expected={expected}")


In [ ]:
# ============================================================
#  DATASET PREPARATION (Training Data)
# ============================================================

def unpack_dataset(raw_data: List[Dict]) -> List[Dict]:
    """Mỗi (index × question) → 1 training sample độc lập."""
    samples = []
    for entry in raw_data:
        fol_list     = entry["premises-FOL"]
        nl_list      = entry["premises-NL"]
        questions    = entry["questions"]
        answers      = entry["answers"]
        explanations = entry["explanation"]
        
        assert len(questions) == len(answers) == len(explanations), (
            f"Mismatch in idx={entry.get('idx')}"
        )
        
        for q, a, exp in zip(questions, answers, explanations):
            user_msg      = make_cot_prompt(fol_list, nl_list, q)
            assistant_msg = build_assistant_message(exp, a)
            samples.append({
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": user_msg},
                    {"role": "assistant", "content": assistant_msg},
                ],
                "answer_label":   a,
                "is_unknown":     a.strip().lower() == "unknown",
                "_raw_entry_idx": id(entry),
            })
    return samples


def apply_adversarial_oversampling(samples, factor=UNKNOWN_OVERSAMPLE_FACTOR, seed=SEED):
    rng = random.Random(seed)
    unknown_s = [s for s in samples if s["is_unknown"]]
    known_s   = [s for s in samples if not s["is_unknown"]]
    combined  = known_s + unknown_s * factor
    rng.shuffle(combined)
    u_count = sum(1 for s in combined if s["is_unknown"])
    print(f"✅ Adversarial Oversampling (x{factor}):")
    print(f"   Total={len(combined)} | Unknown={u_count} ({u_count/len(combined)*100:.1f}%)")
    return combined


def split_dataset(samples, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SEED):
    shuffled = copy.deepcopy(samples)
    random.Random(seed).shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    
    def to_hf(lst):
        return Dataset.from_list([{"messages": s["messages"]} for s in lst])
    
    ds = DatasetDict({
        "train":      to_hf(shuffled[:n_train]),
        "validation": to_hf(shuffled[n_train:n_train+n_val]),
        "test":       to_hf(shuffled[n_train+n_val:]),
    })
    print("📊 Dataset splits:")
    for k, v in ds.items():
        print(f"   {k:12s}: {len(v):5d} samples")
    return ds


flat_samples     = unpack_dataset(raw_data)
augmented_samples = apply_adversarial_oversampling(flat_samples)
dataset           = split_dataset(augmented_samples)
print(f"\n✅ Total flat samples: {len(flat_samples)}")


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Vocab size    : {tokenizer.vocab_size}")
print(f"   Max seq length: {MAX_SEQ_LEN}")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = LORA_TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied:")
print(f"   Trainable : {trainable:,} ({trainable/total_p*100:.2f}%)")
print(f"   Total     : {total_p:,}")


In [ ]:
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
import torch

class TqdmProgressCallback(TrainerCallback):
    def __init__(self):       self.pbar = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="🚀 Training", unit="step", colour="green")
    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.update(1)
            if state.log_history:
                self.pbar.set_postfix({"loss": f"{state.log_history[-1].get('loss',0):.4f}"})
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar: self.pbar.close()


def formatting_prompts_func(examples):
    convos = examples["messages"]
    if convos and isinstance(convos[0], dict):
        convos = [convos]
    return [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    packing                     = False,
    padding_free                = False,
    assistant_only_loss         = False,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_train_epochs            = NUM_EPOCHS,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    seed                        = SEED,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    optim                       = "adamw_8bit",
    push_to_hub                 = False,
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset["train"],
    eval_dataset     = dataset["validation"],
    formatting_func  = formatting_prompts_func,
    args             = training_args,
)

# Verify token markers
sample_formatted = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"], tokenize=False, add_generation_prompt=False,
)
print("Token markers:")
print("  <|im_start|>user      :", "<|im_start|>user" in sample_formatted)
print("  <|im_start|>assistant :", "<|im_start|>assistant" in sample_formatted)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)
print("\n✅ Trainer ready | Train:", len(dataset["train"]), "| Val:", len(dataset["validation"]))


In [ ]:
print("🚀 Starting fine-tuning...")
print("=" * 60)

trainer_stats = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print(f"   Total steps    : {trainer_stats.global_step}")
print(f"   Training time  : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Samples/second : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"   Final loss     : {trainer_stats.metrics['train_loss']:.4f}")
if torch.cuda.is_available():
    print(f"   Peak VRAM      : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


In [ ]:
LORA_SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(LORA_SAVE_PATH, exist_ok=True)

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

saved_files = os.listdir(LORA_SAVE_PATH)
total_size_mb = sum(
    os.path.getsize(os.path.join(LORA_SAVE_PATH, f)) for f in saved_files
) / 1e6

print(f"✅ LoRA adapter saved to: {LORA_SAVE_PATH}")
print(f"   Files: {saved_files}")
print(f"   Size : {total_size_mb:.1f} MB")

# Switch to inference mode
FastLanguageModel.for_inference(model)
print("✅ Switched to inference mode.")


In [ ]:
# ============================================================
#  EVALUATION — Hybrid Z3 + CoT Pipeline trên 411 indices
# ============================================================
print("📊 Evaluating với 5-stage Hybrid Pipeline...")
print("=" * 60)

correct       = 0
total_eval    = 0
unknown_correct = 0
unknown_total = 0
results_log   = []

# Route tracking
route_counts = defaultdict(int)

for idx_i, entry in enumerate(raw_data):
    for q_i, (q, expected_ans, _) in enumerate(
        zip(entry["questions"], entry["answers"], entry["explanation"])
    ):
        # Chạy pipeline
        pred_ans, route = run_hybrid_pipeline(model, tokenizer, entry, q_i)
        
        is_correct = (pred_ans.strip().lower() == expected_ans.strip().lower())
        total_eval += 1
        if is_correct:     correct += 1
        if expected_ans.strip().lower() == "unknown":
            unknown_total += 1
            if is_correct: unknown_correct += 1
        
        route_key = route.split("(")[0]   # strip vote details
        route_counts[route_key] += 1
        
        results_log.append({
            "idx":      idx_i,
            "q_i":      q_i,
            "expected": expected_ans,
            "pred":     pred_ans,
            "correct":  is_correct,
            "route":    route,
        })
        
        # Progress report mỗi 50 indices
        if (idx_i + 1) % 50 == 0 and q_i == 0:
            running_acc = correct / total_eval * 100
            print(f"  [{idx_i+1:3d}/411] Running acc: {running_acc:.1f}% | Routes: {dict(route_counts)}")

print("\n" + "=" * 60)
print("✅ FINAL EVALUATION RESULTS")
print("=" * 60)
overall_acc = correct / total_eval * 100 if total_eval > 0 else 0
print(f"Overall Accuracy  : {correct}/{total_eval} = {overall_acc:.1f}%")

if unknown_total > 0:
    print(f"Unknown Accuracy  : {unknown_correct}/{unknown_total} = {unknown_correct/unknown_total*100:.1f}%")

known_total   = total_eval - unknown_total
known_correct = correct - unknown_correct
if known_total > 0:
    print(f"Known Acc         : {known_correct}/{known_total} = {known_correct/known_total*100:.1f}%")

unparseable = sum(1 for r in results_log if r["pred"] == "UNPARSEABLE")
print(f"Unparseable preds : {unparseable}/{total_eval}")

print(f"\n📊 Route distribution:")
for route, count in sorted(route_counts.items()):
    pct = count/total_eval*100
    print(f"   {route:20s}: {count:4d} ({pct:.1f}%)")

if overall_acc >= 70:
    print(f"\n🎯 TARGET MET: {overall_acc:.1f}% ≥ 70% ✅")
elif overall_acc >= 60:
    print(f"\n⚠️  Baseline target met: {overall_acc:.1f}% ≥ 60% (target 70%)")
else:
    print(f"\n⚠️  Target not met: {overall_acc:.1f}%")


In [ ]:
# ============================================================
#  ERROR ANALYSIS
# ============================================================
wrong_samples = [r for r in results_log if not r["correct"]]
print(f"\n🔍 Error Analysis ({len(wrong_samples)} wrong / {total_eval} total)")
print("=" * 50)

# Confusion matrix
confusion = defaultdict(Counter)
for r in results_log:
    confusion[r["expected"]][r["pred"]] += 1

print("\nConfusion (expected → predicted):")
for expected_label in sorted(confusion.keys()):
    row = confusion[expected_label]
    total_row = sum(row.values())
    correct_count = row.get(expected_label, 0)
    acc_row = correct_count / total_row * 100
    print(f"  {expected_label:8s} (n={total_row:3d}, acc={acc_row:.0f}%): ", end="")
    for pred_label, count in sorted(row.items()):
        marker = "✅" if pred_label == expected_label else "❌"
        print(f"{marker}{pred_label}={count}", end="  ")
    print()

# Route accuracy breakdown
print("\n📊 Accuracy by route:")
route_acc = defaultdict(lambda: [0, 0])   # [correct, total]
for r in results_log:
    route_key = r["route"].split("(")[0]
    route_acc[route_key][1] += 1
    if r["correct"]:
        route_acc[route_key][0] += 1

for route, (c, t) in sorted(route_acc.items()):
    print(f"  {route:20s}: {c}/{t} = {c/t*100:.1f}%")

# Mẫu sai đầu tiên mỗi loại
print(f"\n--- 3 mẫu sai (first failures) ---")
for r in wrong_samples[:3]:
    entry = raw_data[r["idx"]]
    q = entry["questions"][r["q_i"]]
    print(f"  idx={r['idx']}, q={r['q_i']} | Route: {r['route'][:40]}")
    print(f"  Q: {q[:80]}...")
    print(f"  Expected: {r['expected']} | Predicted: {r['pred']}")
    print()


In [ ]:
# ============================================================
#  LƯU KẾT QUẢ & PUSH TO HUB (tùy chọn)
# ============================================================

# Lưu results log
results_path = os.path.join(OUTPUT_DIR, "evaluation_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump({
        "overall_accuracy": overall_acc,
        "correct": correct,
        "total": total_eval,
        "route_distribution": dict(route_counts),
        "results": results_log,
    }, f, ensure_ascii=False, indent=2)
print(f"✅ Results saved to: {results_path}")

# Optional: Push lên HuggingFace Hub
PUSH_TO_HUB = False
HF_REPO_ID  = "Barakuga/qwen3-8b-logic-lora-z3"

if PUSH_TO_HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")
    
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        print(f"✅ Pushed to: https://huggingface.co/{HF_REPO_ID}")
    else:
        print("⚠️  HF_TOKEN not found.")
else:
    print("ℹ️  PUSH_TO_HUB=False. Set True to push to HuggingFace Hub.")
